In [ ]:
import os

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve, roc_curve, roc_auc_score, average_precision_score
from sklearn.inspection import permutation_importance

import seaborn as sns
import matplotlib.pyplot as plt

import joblib

RANDOM_SEED = 42
IMAGE_DIR = "../../results/images/LogisticRegression"
DATASET_DIR = "../../data/processed/supervised/"

os.makedirs(IMAGE_DIR, exist_ok=True)

### Load Data and get Labels

In [ ]:
train_df = pd.read_csv(f'{DATASET_DIR}/train_data.csv')
val_df = pd.read_csv(f'{DATASET_DIR}/val_data.csv')
test_df = pd.read_csv(f'{DATASET_DIR}/test_data.csv')

In [ ]:
X_train = train_df.drop('fraud', axis=1)
y_train = train_df['fraud']

X_val = val_df.drop('fraud', axis=1)
y_val = val_df['fraud']

X_test = test_df.drop('fraud', axis=1)
y_test = test_df['fraud']

### Cross-search Validation

Going through different values of C to see where the models performs best

C is how much the model tries to fit the training data

In [ ]:
results = []

for C in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(C=C, 
                               max_iter=1000,
                               solver="lbfgs",
                               class_weight={0: 1, 1: 10},
                               random_state=RANDOM_SEED)
    
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    probs_val = model.predict_proba(X_val)[:, 1]

    acc = accuracy_score(y_val, y_pred)
    perc = precision_score(y_val, y_pred, zero_division=0)
    rec = recall_score(y_val, y_pred, zero_division=0)
    f1 = f1_score(y_val, y_pred, zero_division=0)

    roc_auc = roc_auc_score(y_val, probs_val)
    pr_auc = average_precision_score(y_val, probs_val)

    results.append({
        "C": C,
        "accuracy": acc,
        "precision": perc,
        "recall": rec,
        "f1_score": f1,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc
        })


In [ ]:
# Best model is decided by precision-recall
results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by="pr_auc", ascending=False)

results_df

In [ ]:
best_results = results_df.iloc[0]

print("Validation C:", best_results["C"])
print("Validation accuracy:", best_results["accuracy"])
print("Validation precision:", best_results["precision"])
print("Validation recall:", best_results["recall"])
print("Validation f1:", best_results["f1_score"])
print("Validation ROC AUC:", best_results["roc_auc"])
print("Validation PR AUC:", best_results["pr_auc"])

### Threshold tuning on Validation set

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, probs_val)

plt.plot(precision, recall, label=f'Logistic Regression (PR AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Validation)')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_validation.svg')
plt.show()

In [ ]:
beta = 2    # Makes F2 score, puts more emphasis on recall
fbeta_scores = (1 + beta**2) * (precision * recall) / (beta**2 * precision + recall + 1e-8)

best_idx = fbeta_scores.argmax()
best_threshold = thresholds[best_idx]

### Test the model

In [ ]:
train_val_df = pd.concat([train_df, val_df], ignore_index=True)

# Final fit on train & val data
X_train_val = train_val_df.drop('fraud', axis=1)
y_train_val = train_val_df['fraud']

model_final = LogisticRegression(
    C=best_results["C"],
    max_iter=1000,
    solver="lbfgs",
    class_weight={0: 1, 1: 10},
    random_state=RANDOM_SEED)

model_final.fit(X_train_val, y_train_val)
joblib.dump({"model": model_final, "threshold": best_threshold}, "../../results/models/logistic_regression.pkl")

probs = model_final.predict_proba(X_test)[:, 1]
y_test_pred = (probs > best_threshold).astype(int)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred, zero_division=0)
recall = recall_score(y_test, y_test_pred, zero_division=0)
f1 = f1_score(y_test, y_test_pred, zero_division=0)

pr_auc = average_precision_score(y_test, probs)
roc_auc = roc_auc_score(y_test, probs)

print("Test accuracy:", accuracy)
print("Test precision:", precision)
print("Test recall:", recall)
print("Test f1:", f1)

print("Test PR AUC:", pr_auc)
print("Test ROC AUC:", roc_auc)

In [ ]:
cf_matrix = confusion_matrix(y_test, y_test_pred)

group_names = ["True Negatives", "False Positives", "False Negatives", "True Positives"]

group_counts = ["{0:0.0f}".format(value) for value in cf_matrix.flatten()]

labels = [f"{v1}\n{v2}" for v1, v2 in zip(group_names,group_counts)]
labels = np.array(labels).reshape(2,2)

matrix = sns.heatmap(cf_matrix, annot=labels, fmt='', cmap="viridis", cbar=False)
matrix.set_xlabel("Predicted Label")
matrix.set_ylabel("True Label")
matrix.set_xticklabels(["Negative", "Positive"])
matrix.set_yticklabels(["Negative", "Positive"])
plt.title('Confusion Matrix')
plt.savefig(f"{IMAGE_DIR}/confusion_matrix.svg")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probs)

plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {roc_auc:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/roc_curve_test.svg')
plt.show()

In [ ]:
precision, recall, _ = precision_recall_curve(y_test, probs)

plt.plot(precision, recall, label=f'Logistic Regression (AUC = {pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(f'{IMAGE_DIR}/pr_curve_test.svg')
plt.show()

### Feature Coeffecients

How features are used to decide if an input is fraud or not

- Positive pushes it more towards it being fraud
- Negative pusges it more toward it being non-fraud

In [ ]:

coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": model_final.coef_[0]
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values(by="abs_coefficient", ascending=False)

top_coefs = coef_df.head(15).sort_values(by="coefficient", ascending=True)

plt.figure(10, 6)
plt.barh(top_coefs["feature"], top_coefs["coefficient"])
plt.xlabel("Coefficient Value")
plt.title("Top 15 Feature Coefficients")
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/feature_importance.svg')
plt.show()



### Feature Permuatation
What features from training does the model consider would decrease/increase performance if they were removed

In [ ]:
result = permutation_importance(
    model_final, 
    X_test, 
    y_test, 
    n_repeats=10, 
    random_state=RANDOM_SEED,
    n_jobs=-1
)

perm_imp = pd.Series(result.importances_mean, index=X_test.columns)
perm_imp = perm_imp.sort_values(ascending=False)

plt.figure(figsize=(10, 6))
perm_imp.sort_values().plot(kind='barh')
plt.title("Permutation Feature Importances")
plt.xlabel("Decrease in Model Performance")
plt.tight_layout()
plt.savefig(f'{IMAGE_DIR}/permutation_importance.svg')
plt.show()